# 2. EDA

In [1]:
!pip install plotly

In [2]:
!pip install "nbformat>=4.2.0"

In [28]:
import pandas as pd
import plotly.express as px
import plotly.io as pio

# 1. Plotly 렌더러 설정
pio.renderers.default = "notebook"
pio.templates.default = "plotly_white"

# 2. 파일 로드
excel_path = '/Users/lusia/Personal/13. DA Bootcamp/04. Project/2. ESG/01_데이터셋/260827_전기전자업 전체 업종.xlsx'
csv_path = '/Users/lusia/Personal/13. DA Bootcamp/04. Project/2. ESG/01_데이터셋/KCGS_평가등급_2022-2025_Long_merged.csv'

df_excel = pd.read_excel(excel_path, header=None)
df_excel.columns = ['기업명', '기업코드']

df_kcgs = pd.read_csv(csv_path)

# 문자열 공백 제거 정제
df_excel['기업명'] = df_excel['기업명'].astype(str).str.strip()
df_kcgs['기업명'] = df_kcgs['기업명'].astype(str).str.strip()
if '업종명' in df_kcgs.columns:
    df_kcgs['업종명'] = df_kcgs['업종명'].astype(str).str.strip()

# ----------------------------------------------------
# [핵심] 101개 전기·전자 Target 기업 데이터만 정확히 필터링
# ----------------------------------------------------
df_elec = pd.merge(df_kcgs, df_excel[['기업명']], on='기업명', how='inner')

# ----------------------------------------------------
# [핵심] 2025년 기준 데이터 추출 (2025년 없으면 2024년 최신값으로 자동 보완)
# ----------------------------------------------------
df_elec_sorted = df_elec.sort_values('평가년도')
df_2025 = df_elec_sorted.groupby('기업명').last().reset_index()

grade_order = ['A+', 'A', 'B+', 'B', 'C', 'D']
df_2025['ESG등급'] = pd.Categorical(df_2025['ESG등급'], categories=grade_order, ordered=True)

# ----------------------------------------------------
# [EDA 1] 2025년 기준 전기·전자 101개 기업 ESG 등급 분포
# ----------------------------------------------------
total_summary_2025 = df_2025['ESG등급'].value_counts().reindex(grade_order).fillna(0).reset_index()
total_summary_2025.columns = ['ESG등급', '기업수']
total_summary_2025['기업수'] = total_summary_2025['기업수'].astype(int)
total_summary_2025['비중(%)'] = (total_summary_2025['기업수'] / total_summary_2025['기업수'].sum() * 100).round(1)

print("=== 2025년 기준 전기·전자 Target 기업 ESG 등급 분포 ===")
print(total_summary_2025)
print(f"총 분석 기업 수: {total_summary_2025['기업수'].sum()}개사")

color_map = {
    'A+': '#1D3557', 'A': '#457B9D', 'B+': '#A8DADC',
    'B': '#F4A261', 'C': '#E76F51', 'D': '#E63946'
}

fig1 = px.bar(
    total_summary_2025,
    x='ESG등급',
    y='기업수',
    text='비중(%)',
    title=f'<b>[전기·전자 Target 기업] 2025년 기준 ESG 등급 분포 (총 {total_summary_2025["기업수"].sum()}개사)</b>',
    labels={'기업수': '기업 수 (개사)', 'ESG등급': 'ESG 통합 등급'},
    color='ESG등급',
    color_discrete_map=color_map
)

fig1.update_traces(texttemplate='%{y}개사<br>(%{text:.1f}%)', textposition='outside')
fig1.update_layout(
    font_family='Apple SD Gothic Neo',
    yaxis=dict(range=[0, total_summary_2025['기업수'].max() * 1.3]),
    showlegend=False
)

fig1.show()

# ----------------------------------------------------
# [EDA 2] 전기·전자 Target 기업 연도별 ESG 등급 추이 (2022-2025)
# ----------------------------------------------------
# 연도별로 정확히 1개 기업당 1개 데이터만 집계
df_elec['ESG등급'] = pd.Categorical(df_elec['ESG등급'], categories=grade_order, ordered=True)
yearly_summary = df_elec.groupby(['평가년도', 'ESG등급'], observed=False).size().reset_index(name='기업수')

fig2 = px.bar(
    yearly_summary,
    x='평가년도',
    y='기업수',
    color='ESG등급',
    title='<b>[전기·전자 Target 기업] 연도별 ESG 등급 분포 추이 (2022-2025)</b>',
    labels={'기업수': '기업 수 (개사)', '평가년도': '평가년도'},
    color_discrete_map=color_map
)

fig2.update_layout(
    font_family='Apple SD Gothic Neo',
    barmode='stack',
    xaxis=dict(type='category')
)

fig2.show()

=== 2025년 기준 전기·전자 Target 기업 ESG 등급 분포 ===
  ESG등급  기업수  비중(%)
0    A+    1    1.0
1     A   19   18.8
2    B+   14   13.9
3     B    8    7.9
4     C   20   19.8
5     D   39   38.6
총 분석 기업 수: 101개사


In [10]:
import pandas as pd
import plotly.express as px
import plotly.io as pio

# Plotly 렌더러 설정 (노트북 환경용)
pio.renderers.default = "notebook"
pio.templates.default = "plotly_white"

# 1. 엑셀 파일 로드 (101개 전기전자 기업 명단)
excel_path = '/Users/lusia/Personal/13. DA Bootcamp/04. Project/2. ESG/01_데이터셋/260827_전기전자업 전체 업종.xlsx'
df_excel = pd.read_excel(excel_path, header=None)
df_excel.columns = ['기업명', '기업코드']

# 2. KCGS ESG 데이터셋 로드
csv_path = '/Users/lusia/Personal/13. DA Bootcamp/04. Project/2. ESG/01_데이터셋/KCGS_평가등급_2022-2025_Long_merged.csv'
df_kcgs = pd.read_csv(csv_path)

# 3. 6대 세부 섹터 분류 규칙 정의
def assign_sector(company_name):
    semicon = ['DB하이텍', 'KEC', 'SK하이닉스', '삼성전자', '네패스', 'SF반도체', '해성디에스', '하나마이크론', '어보브반도체', '시그네틱스', '제주반도체', '텔레칩스', '칩스앤미디어', '리노공업', '동진쎄미켐', 'SOLUM', '솔브레인']
    display = ['LG디스플레이', 'LX세미콘', '일진디스플', '파워넷', '선익시스템', '야스', '인포뱅크', '아이씨디', '덕산네오룩스', '코세스']
    battery_auto = ['LG에너지솔루션', '삼성SDI', 'SK아이이테크놀로지', '엘앤에프', '포스코퓨처엠', '에코프로비엠', 'DN오토모티브', 'LS머트리얼즈', '모베이스전자', '현대모비스', 'HL만도']
    power_cable = ['HD현대일렉트릭', 'LS일렉트릭', 'LS전선', '대한전선', '가온전선', '일진전기', '효성중공업', '제룡전기', '광명전기', '보성파워텍', '서전기전', 'HD현대에너지솔루션', 'KH 필룩스']
    home_app = ['LG전자', '위니아', '쿠쿠홈시스', '쿠쿠홀딩스', '신일전자', '파セ', '오텍', '위닉스', 'PN풍년']
    
    if any(c in company_name for c in semicon): return '1. 반도체'
    if any(c in company_name for c in display): return '2. 디스플레이'
    if any(c in company_name for c in battery_auto): return '3. 이차전지 및 전장'
    if any(c in company_name for c in power_cable): return '4. 전력기기 및 전선'
    if any(c in company_name for c in home_app): return '5. 가전 및 가전제품'
    return '6. IT 장비 및 부품'

df_excel['세부섹터'] = df_excel['기업명'].apply(assign_sector)

# 4. 101개 기업 목록과 KCGS ESG 평가 데이터 결합
df_target_101 = pd.merge(df_kcgs, df_excel[['기업명', '세부섹터']], on='기업명', how='inner')

# 5. ESG 등급 정렬
grade_order = ['A+', 'A', 'B+', 'B', 'C', 'D']
df_target_101['ESG등급'] = pd.Categorical(df_target_101['ESG등급'], categories=grade_order, ordered=True)

# ----------------------------------------------------
# [EDA 결과 출력]
# ----------------------------------------------------
print("=== 1. 엑셀 기준 101개 기업의 세부 섹터별 고유 회사 수 ===")
sector_company_counts = df_excel.groupby('세부섹터')['기업명'].nunique().reset_index(name='회사수')
print(sector_company_counts)
print(f"\n총 포함 회사 수: {sector_company_counts['회사수'].sum()}개사")

=== 1. 엑셀 기준 101개 기업의 세부 섹터별 고유 회사 수 ===
            세부섹터  회사수
0         1. 반도체   11
1       2. 디스플레이    4
2   3. 이차전지 및 전장    8
3   4. 전력기기 및 전선    9
4   5. 가전 및 가전제품    1
5  6. IT 장비 및 부품   68

총 포함 회사 수: 101개사


In [26]:
# ----------------------------------------------------
# [EDA] 세부 섹터별 총 기업 수 및 매칭 현황 분석
# ----------------------------------------------------

# 1. 섹터별 총 기업 수 및 비율 산출
sector_summary = df_excel.groupby('세부섹터')['기업명'].nunique().reset_index(name='전체 기업 수')
sector_summary['비중(%)'] = (sector_summary['전체 기업 수'] / sector_summary['전체 기업 수'].sum() * 100).round(1)

# 2. KCGS ESG 데이터셋과 정상 매칭된 기업 수 확인
matched_counts = df_target_101.groupby('세부섹터')['기업명'].nunique().reset_index(name='ESG데이터 보유 기업 수')

# 3. 종합 현황 데이터프레임 생성
eda_sector_df = pd.merge(sector_summary, matched_counts, on='세부섹터', how='left').fillna(0)
eda_sector_df['ESG데이터 보유 기업 수'] = eda_sector_df['ESG데이터 보유 기업 수'].astype(int)

# 4. 섹터별 대표 기업 (최대 5개사) 샘플 추출
def get_sample_companies(sector):
    companies = df_excel[df_excel['세부섹터'] == sector]['기업명'].unique()
    return ", ".join(companies[:5]) + ("..." if len(companies) > 5 else "")

eda_sector_df['주요 기업 샘플'] = eda_sector_df['세부섹터'].apply(get_sample_companies)

# ----------------------------------------------------
# [결과 출력]
# ----------------------------------------------------
print("="*80)
print("■ 1. 6대 세부 섹터별 기업 수 요약")
print("="*80)
print(eda_sector_df.to_string(index=False))

print("\n" + "="*80)
print(f"■ 2. 전체 요약: 총 {df_excel['기업명'].nunique()}개사 중 {df_target_101['기업명'].nunique()}개사 ESG 데이터 매칭 완료")
print("="*80)


# ----------------------------------------------------
# (선택) Plotly 시각화: 섹터별 기업 수 분포
# ----------------------------------------------------
fig_sector = px.bar(
    eda_sector_df,
    x='세부섹터',
    y='전체 기업 수',
    text='전체 기업 수',
    color='세부섹터',
    title='<b>전기전자업 6대 세부 섹터별 기업 수 분포</b>',
    labels={'세부섹터': '세부 섹터', '전체 기업 수': '기업 수 (개사)'},
    color_discrete_sequence=px.colors.qualitative.Pastel
)

fig_sector.update_layout(
    font_family='Apple SD Gothic Neo',
    showlegend=False,
    yaxis=dict(range=[0, eda_sector_df['전체 기업 수'].max() * 1.15]),
    margin=dict(l=40, r=40, t=60, b=40)
)
fig_sector.update_traces(textposition='outside')
fig_sector.show()

■ 1. 6대 세부 섹터별 기업 수 요약
         세부섹터  전체 기업 수  비중(%)  ESG데이터 보유 기업 수                                         주요 기업 샘플
       1. 반도체       11   10.9              11                 DB하이텍, KEC, SK하이닉스, 네패스, 리노공업...
     2. 디스플레이        4    4.0               4                    LG디스플레이, LX세미콘, 덕산네오룩스, 일진디스플
 3. 이차전지 및 전장        8    7.9               8 DN오토모티브, LG에너지솔루션, LS머트리얼즈, SK아이이테크놀로지, 삼성SDI...
 4. 전력기기 및 전선        9    8.9               9      HD현대에너지솔루션, HD현대일렉트릭, KH 필룩스, 가온전선, 광명전기...
 5. 가전 및 가전제품        1    1.0               1                                             LG전자
6. IT 장비 및 부품       68   67.3              68             LG이노텍, RFHIC, SFA반도체, 가온칩스, 경동나비엔...

■ 2. 전체 요약: 총 101개사 중 101개사 ESG 데이터 매칭 완료


In [27]:
import pandas as pd
import plotly.express as px
import plotly.io as pio

# Plotly 렌더러 설정 (노트북 환경용)
pio.renderers.default = "notebook"
pio.templates.default = "plotly_white"

# 파일 로드 및 결합
df_excel = pd.read_excel('/Users/lusia/Personal/13. DA Bootcamp/04. Project/2. ESG/01_데이터셋/260827_전기전자업 전체 업종.xlsx', header=None)
df_excel.columns = ['기업명', '기업코드']
df_kcgs = pd.read_csv('/Users/lusia/Personal/13. DA Bootcamp/04. Project/2. ESG/01_데이터셋/KCGS_평가등급_2022-2025_Long_merged.csv')

def assign_sector(company_name):
    semicon = ['DB하이텍', 'KEC', 'SK하이닉스', '삼성전자', '네패스', 'SF반도체', '해성디에스', '하나마이크론', '어보브반도체', '시그네틱스', '제주반도체', '텔레칩스', '칩스앤미디어', '리노공업', '동진쎄미켐', 'SOLUM', '솔브레인']
    display = ['LG디스플레이', 'LX세미콘', '일진디스플', '파워넷', '선익시스템', '야스', '인포뱅크', '아이씨디', '덕산네오룩스', '코세스']
    battery_auto = ['LG에너지솔루션', '삼성SDI', 'SK아이이테크놀로지', '엘앤에프', '포스코퓨처엠', '에코프로비엠', 'DN오토모티브', 'LS머트리얼즈', '모베이스전자', '현대모비스', 'HL만도']
    power_cable = ['HD현대일렉트릭', 'LS일렉트릭', 'LS전선', '대한전선', '가온전선', '일진전기', '효성중공업', '제룡전기', '광명전기', '보성파워텍', '서전기전', 'HD현대에너지솔루션', 'KH 필룩스']
    home_app = ['LG전자', '위니아', '쿠쿠홈시스', '쿠쿠홀딩스', '신일전자', '파セ', '오텍', '위닉스', 'PN풍년']

    if any(c in company_name for c in semicon): return '1. 반도체'
    if any(c in company_name for c in display): return '2. 디스플레이'
    if any(c in company_name for c in battery_auto): return '3. 이차전지 및 전장'
    if any(c in company_name for c in power_cable): return '4. 전력기기 및 전선'
    if any(c in company_name for c in home_app): return '5. 가전 및 가전제품'
    return '6. IT 장비 및 부품'

df_excel['세부섹터'] = df_excel['기업명'].apply(assign_sector)
df_merged = pd.merge(df_kcgs, df_excel[['기업명', '세부섹터']], on='기업명', how='inner')

In [13]:
import pandas as pd
import plotly.express as px
import plotly.io as pio

pio.renderers.default = "notebook"
pio.templates.default = "plotly_white"

# 선택된 4개 섹터 데이터 필터링 (최신 평가 기준)
target_sectors = ['1. 반도체', '2. 디스플레이', '4. 전력기기 및 전선', '6. IT 장비 및 부품']
df_filtered = df_merged[df_merged['세부섹터'].isin(target_sectors)].copy()

# 기업별 최신 연도 평가 기준
latest_eval = df_filtered.sort_values('평가년도').groupby(['세부섹터', '기업명']).last().reset_index()

grade_order = ['A+', 'A', 'B+', 'B', 'C', 'D']
latest_eval['ESG등급'] = pd.Categorical(latest_eval['ESG등급'], categories=grade_order, ordered=True)

# 집계
summary_df = latest_eval.groupby(['세부섹터', 'ESG등급'], observed=False).size().reset_index(name='기업수')

# 막대그래프 생성
fig = px.bar(
    summary_df,
    x='세부섹터',
    y='기업수',
    color='ESG등급',
    barmode='group',
    text='기업수',
    title='<b>4개 주요 섹터별 ESG 등급 분포 (기업 수 기준)</b>',
    labels={'기업수': '기업 수 (개사)', '세부섹터': '세부 섹터'},
    color_discrete_map={
        'A+': '#1D3557', 'A': '#457B9D', 'B+': '#A8DADC',
        'B': '#F4A261', 'C': '#E76F51', 'D': '#E63946'
    }
)

fig.update_traces(textposition='outside')
fig.update_layout(font_family='Apple SD Gothic Neo', yaxis=dict(range=[0, summary_df['기업수'].max() * 1.15]))
fig.show()

In [18]:
import pandas as pd
import plotly.express as px
import plotly.io as pio

# 1. Plotly 렌더러 설정 (노트북용)
pio.renderers.default = "notebook"
pio.templates.default = "plotly_white"

# 2. 파일 로드
excel_path = '/Users/lusia/Personal/13. DA Bootcamp/04. Project/2. ESG/01_데이터셋/260827_전기전자업 전체 업종.xlsx'
csv_path = '/Users/lusia/Personal/13. DA Bootcamp/04. Project/2. ESG/01_데이터셋/KCGS_평가등급_2022-2025_Long_merged.csv'

df_excel = pd.read_excel(excel_path, header=None)
df_excel.columns = ['기업명', '기업코드']
df_kcgs = pd.read_csv(csv_path)

# 3. 세부 섹터 분류 규칙
def assign_sector(company_name):
    semicon = ['DB하이텍', 'KEC', 'SK하이닉스', '삼성전자', '네패스', 'SF반도체', '해성디에스', '하나마이크론', '어보브반도체', '시그네틱스', '제주반도체', '텔레칩스', '칩스앤미디어', '리노공업', '동진쎄미켐', 'SOLUM', '솔브레인']
    display = ['LG디스플레이', 'LX세미콘', '일진디스플', '파워넷', '선익시스템', '야스', '인포뱅크', '아이씨디', '덕산네오룩스', '코세스']
    battery_auto = ['LG에너지솔루션', '삼성SDI', 'SK아이이테크놀로지', '엘앤에프', '포스코퓨처엠', '에코프로비엠', 'DN오토모티브', 'LS머트리얼즈', '모베이스전자', '현대모비스', 'HL만도']
    power_cable = ['HD현대일렉트릭', 'LS일렉트릭', 'LS전선', '대한전선', '가온전선', '일진전기', '효성중공업', '제룡전기', '광명전기', '보성파워텍', '서전기전', 'HD현대에너지솔루션', 'KH 필룩스']
    home_app = ['LG전자', '위니아', '쿠쿠홈시스', '쿠쿠홀딩스', '신일전자', '파セ', '오텍', '위닉스', 'PN풍년']

    if any(c in company_name for c in semicon): return '1. 반도체'
    if any(c in company_name for c in display): return '2. 디스플레이'
    if any(c in company_name for c in battery_auto): return '3. 이차전지 및 전장'
    if any(c in company_name for c in power_cable): return '4. 전력기기 및 전선'
    if any(c in company_name for c in home_app): return '5. 가전 및 가전제품'
    return '6. IT 장비 및 부품'

df_excel['세부섹터'] = df_excel['기업명'].apply(assign_sector)

# 4. 데이터 결합 및 최신 평가 데이터 추출
df_merged = pd.merge(df_kcgs, df_excel[['기업명', '세부섹터']], on='기업명', how='inner')
latest_eval = df_merged.sort_values('평가년도').groupby(['세부섹터', '기업명']).last().reset_index()

# 5. 등급 순서 및 서열 점수 매핑 (시각화 축 정렬용)
grade_order = ['A+', 'A', 'B+', 'B', 'C', 'D']
grade_score = {'A+': 6, 'A': 5, 'B+': 4, 'B': 3, 'C': 2, 'D': 1}

latest_eval['ESG등급'] = pd.Categorical(latest_eval['ESG등급'], categories=grade_order, ordered=True)
latest_eval['등급점수'] = latest_eval['ESG등급'].map(grade_score)

# 색상 매핑
color_map = {
    'A+': '#1D3557', 'A': '#457B9D', 'B+': '#A8DADC',
    'B': '#F4A261', 'C': '#E76F51', 'D': '#E63946'
}

# ------------------------------------------------------------------
# [각 섹터별 기업 전체 등급순 나열 시각화]
# ------------------------------------------------------------------
target_sectors = ['1. 반도체', '2. 디스플레이', '4. 전력기기 및 전선', '6. IT 장비 및 부품']

for sector_name in target_sectors:
    # 해당 섹터 데이터를 등급높은순 -> 기업명순으로 정렬
    sec_df = latest_eval[latest_eval['세부섹터'] == sector_name].sort_values(['ESG등급', '기업명']).copy()
    
    # 개별 섹터 내 기업별 등급 막대그래프
    fig = px.bar(
        sec_df,
        x='기업명',
        y='등급점수',
        color='ESG등급',
        text='ESG등급',
        title=f'<b>[{sector_name}] 전체 기업 ESG 등급순 나열 (총 {len(sec_df)}개사)</b>',
        labels={'기업명': '기업명', '등급점수': 'ESG 등급 순위', 'ESG등급': 'ESG 등급'},
        color_discrete_map=color_map,
        category_orders={'ESG등급': grade_order}
    )
    
    # y축 눈금을 숫자가 아닌 실제 등급 문자(A+ ~ D)로 대체
    fig.update_layout(
        font_family='Apple SD Gothic Neo',
        yaxis=dict(
            tickmode='array',
            tickvals=[1, 2, 3, 4, 5, 6],
            ticktext=['D', 'C', 'B', 'B+', 'A', 'A+'],
            range=[0, 7]
        ),
        xaxis=dict(tickangle=-45),  # 기업명이 겹치지 않도록 사선 처리
        showlegend=True
    )
    
    fig.update_traces(textposition='outside')
    
    # 개별 차트 출력
    fig.show()

In [20]:
import pandas as pd
import plotly.express as px
import plotly.io as pio

# 1. Plotly 렌더러 설정
pio.renderers.default = "notebook"
pio.templates.default = "plotly_white"

# 2. 파일 로드
excel_path = '/Users/lusia/Personal/13. DA Bootcamp/04. Project/2. ESG/01_데이터셋/260827_전기전자업 전체 업종.xlsx'
csv_path = '/Users/lusia/Personal/13. DA Bootcamp/04. Project/2. ESG/01_데이터셋/KCGS_평가등급_2022-2025_Long_merged.csv'

df_excel = pd.read_excel(excel_path, header=None)
df_excel.columns = ['기업명', '기업코드']
df_kcgs = pd.read_csv(csv_path)

# 3. 세부 섹터 분류 함수
def assign_sector(company_name):
    semicon = ['DB하이텍', 'KEC', 'SK하이닉스', '삼성전자', '네패스', 'SF반도체', '해성디에스', '하나마이크론', '어보브반도체', '시그네틱스', '제주반도체', '텔레칩스', '칩스앤미디어', '리노공업', '동진쎄미켐', 'SOLUM', '솔브레인']
    display = ['LG디스플레이', 'LX세미콘', '일진디스플', '파워넷', '선익시스템', '야스', '인포뱅크', '아이씨디', '덕산네오룩스', '코세스']
    battery_auto = ['LG에너지솔루션', '삼성SDI', 'SK아이이테크놀로지', '엘앤에프', '포스코퓨처엠', '에코프로비엠', 'DN오토모티브', 'LS머트리얼즈', '모베이스전자', '현대모비스', 'HL만도']
    power_cable = ['HD현대일렉트릭', 'LS일렉트릭', 'LS전선', '대한전선', '가온전선', '일진전기', '효성중공업', '제룡전기', '광명전기', '보성파워텍', '서전기전', 'HD현대에너지솔루션', 'KH 필룩스']
    home_app = ['LG전자', '위니아', '쿠쿠홈시스', '쿠쿠홀딩스', '신일전자', '파セ', '오텍', '위닉스', 'PN풍년']

    if any(c in company_name for c in semicon): return '1. 반도체'
    if any(c in company_name for c in display): return '2. 디스플레이'
    if any(c in company_name for c in battery_auto): return '3. 이차전지 및 전장'
    if any(c in company_name for c in power_cable): return '4. 전력기기 및 전선'
    if any(c in company_name for c in home_app): return '5. 가전 및 가전제품'
    return '6. IT 장비 및 부품'

df_excel['세부섹터'] = df_excel['기업명'].apply(assign_sector)

# 4. 데이터 결합 및 최신 평가 추출
df_merged = pd.merge(df_kcgs, df_excel[['기업명', '세부섹터']], on='기업명', how='inner')
latest_eval = df_merged.sort_values('평가년도').groupby(['세부섹터', '기업명']).last().reset_index()

# 5. 등급 순서 및 서열 점수 매핑
grade_order = ['A+', 'A', 'B+', 'B', 'C', 'D']
grade_score = {'A+': 6, 'A': 5, 'B+': 4, 'B': 3, 'C': 2, 'D': 1}

latest_eval['ESG등급'] = pd.Categorical(latest_eval['ESG등급'], categories=grade_order, ordered=True)

# 6. 타겟 섹터 설정 및 데이터 변환 (Melt)
target_sectors = ['1. 반도체', '6. IT 장비 및 부품']

for sector_name in target_sectors:
    sec_df = latest_eval[latest_eval['세부섹터'] == sector_name].copy()
    
    # 우수 기업(A 이상)과 최하위 5개사 필터링
    top_df = sec_df[sec_df['ESG등급'].isin(['A+', 'A'])].sort_values(['ESG등급', '기업명'])
    bottom_df = sec_df.sort_values(['ESG등급', '기업명'], ascending=[False, True]).head(5)
    
    selected_df = pd.concat([top_df, bottom_df]).drop_duplicates()
    
    # E, S, G 열을 긴 형태(Long-format)로 변환
    melted = pd.melt(
        selected_df,
        id_vars=['기업명', 'ESG등급'],
        value_vars=['환경', '사회', '지배구조'],
        var_name='평가요소',
        value_name='세부등급'
    )
    
    melted['등급점수'] = melted['세부등급'].map(grade_score)
    melted['세부등급'] = pd.Categorical(melted['세부등급'], categories=grade_order, ordered=True)

    # ------------------------------------------------------------------
    # Plotly 시각화 (그룹 막대 그래프)
    # ------------------------------------------------------------------
    fig = px.bar(
        melted,
        x='기업명',
        y='등급점수',
        color='평가요소',
        barmode='group',
        text='세부등급',
        title=f'<b>[{sector_name}] 우수 기업 및 최하위 5개사 E/S/G 세부 등급 비교</b>',
        labels={'기업명': '기업명', '등급점수': '등급 서열', '평가요소': 'ESG 요소'},
        color_discrete_sequence=['#2A9D8F', '#E76F51', '#264653']  # E: 청록, S: 주황, G: 남색
    )

    # y축을 숫자 대신 실제 등급 표기로 전환
    fig.update_layout(
        font_family='Apple SD Gothic Neo',
        yaxis=dict(
            tickmode='array',
            tickvals=[1, 2, 3, 4, 5, 6],
            ticktext=['D', 'C', 'B', 'B+', 'A', 'A+'],
            range=[0, 7]
        ),
        xaxis=dict(tickangle=-30),
        legend=dict(title="평가 요소"),
        margin=dict(l=40, r=40, t=60, b=80)
    )

    fig.update_traces(textposition='outside')
    
    # 개별 차트 출력
    fig.show()

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.io as pio

# 1. Plotly 렌더러 설정
pio.renderers.default = "notebook"
pio.templates.default = "plotly_white"

# 2. 파일 로드
excel_path = '/Users/lusia/Personal/13. DA Bootcamp/04. Project/2. ESG/01_데이터셋/260827_전기전자업 전체 업종.xlsx'
csv_path = '/Users/lusia/Personal/13. DA Bootcamp/04. Project/2. ESG/01_데이터셋/KCGS_평가등급_2022-2025_Long_merged.csv'

df_excel = pd.read_excel(excel_path, header=None)
df_excel.columns = ['기업명', '기업코드']
df_kcgs = pd.read_csv(csv_path)

# 3. 세부 섹터 분류 함수
def assign_sector(company_name):
    semicon = ['DB하이텍', 'KEC', 'SK하이닉스', '삼성전자', '네패스', 'SF반도체', '해성디에스', '하나마이크론', '어보브반도체', '시그네틱스', '제주반도체', '텔레칩스', '칩스앤미디어', '리노공업', '동진쎄미켐', 'SOLUM', '솔브레인']
    display = ['LG디스플레이', 'LX세미콘', '일진디스플', '파워넷', '선익시스템', '야스', '인포뱅크', '아이씨디', '덕산네오룩스', '코세스']
    battery_auto = ['LG에너지솔루션', '삼성SDI', 'SK아이이테크놀로지', '엘앤에프', '포스코퓨처엠', '에코프로비엠', 'DN오토모티브', 'LS머트리얼즈', '모베이스전자', '현대모비스', 'HL만도']
    power_cable = ['HD현대일렉트릭', 'LS일렉트릭', 'LS전선', '대한전선', '가온전선', '일진전기', '효성중공업', '제룡전기', '광명전기', '보성파워텍', '서전기전', 'HD현대에너지솔루션', 'KH 필룩스']
    home_app = ['LG전자', '위니아', '쿠쿠홈시스', '쿠쿠홀딩스', '신일전자', '파セ', '오텍', '위닉스', 'PN풍년']

    if any(c in company_name for c in semicon): return '1. 반도체'
    if any(c in company_name for c in display): return '2. 디스플레이'
    if any(c in company_name for c in battery_auto): return '3. 이차전지 및 전장'
    if any(c in company_name for c in power_cable): return '4. 전력기기 및 전선'
    if any(c in company_name for c in home_app): return '5. 가전 및 가전제품'
    return '6. IT 장비 및 부품'

df_excel['세부섹터'] = df_excel['기업명'].apply(assign_sector)

# 4. 데이터 결합
df_merged = pd.merge(df_kcgs, df_excel[['기업명', '세부섹터']], on='기업명', how='inner')

# ------------------------------------------------------------------
# [추론 적용 로직] 25년/24년 등 과거 연도 중 1.0이 하나라도 있으면 최종 1.0(발행) 처리
# ------------------------------------------------------------------
def resolve_report_status(series):
    # 숫자형 변환 후 1.0 존재 여부 확인
    numeric_vals = pd.to_numeric(series, errors='coerce')
    if (numeric_vals == 1.0).any():
        return 1.0
    return 0.0

# 기업별 가장 최신 ESG 등급 추출
latest_grade = df_merged.sort_values('평가년도').groupby(['세부섹터', '기업명'])['ESG등급'].last().reset_index()

# 보고서 발행 여부는 전 연도 통합(최대값/1.0 우선) 집계
report_status = df_merged.groupby(['세부섹터', '기업명']).agg({
    '지속가능경영보고서_발행': resolve_report_status,
    '기업지배구조보고서_발행': resolve_report_status
}).reset_index()

# 통합 데이터 병합
final_df = pd.merge(latest_grade, report_status, on=['세부섹터', '기업명'])

grade_order = ['A+', 'A', 'B+', 'B', 'C', 'D']
final_df['ESG등급'] = pd.Categorical(final_df['ESG등급'], categories=grade_order, ordered=True)

# 5. 색상 매핑 설정
color_map = {
    '지속가능경영보고서 (발급)': '#2ECC71',  # 연두색
    '기업지배구조보고서 (발급)': '#1D3557',  # 남색
    '미발급': '#E0E0E0'                     # 회색
}

target_sectors = ['1. 반도체', '6. IT 장비 및 부품']

for sector_name in target_sectors:
    sec_df = final_df[final_df['세부섹터'] == sector_name].copy()
    
    # 우수 기업(A 이상) + 최하위 5개사 추출
    top_df = sec_df[sec_df['ESG등급'].isin(['A+', 'A'])].sort_values(['ESG등급', '기업명'])
    bottom_df = sec_df.sort_values(['ESG등급', '기업명'], ascending=[False, True]).head(5)
    selected_df = pd.concat([top_df, bottom_df]).drop_duplicates()

    # Melt 변환
    melted = pd.melt(
        selected_df,
        id_vars=['기업명', 'ESG등급'],
        value_vars=['지속가능경영보고서_발행', '기업지배구조보고서_발행'],
        var_name='보고서종류',
        value_name='발급여부_raw'
    )
    
    melted['보고서종류'] = melted['보고서종류'].str.replace('_발행', '')
    
    # 발급 상태 텍스트 매핑
    melted['발급여부'] = melted['발급여부_raw'].apply(lambda x: '발급' if x == 1.0 else '미발급')
    melted['발급상태'] = melted.apply(
        lambda r: f"{r['보고서종류']} (발급)" if r['발급여부'] == '발급' else '미발급', 
        axis=1
    )
    melted['표시높이'] = 1

    # ------------------------------------------------------------------
    # Plotly 시각화
    # ------------------------------------------------------------------
    fig = px.bar(
        melted,
        x='기업명',
        y='표시높이',
        color='발급상태',
        facet_col='보고서종류',
        text='발급여부',
        title=f'<b>[{sector_name}] 보고서 2종 발급 현황 (24~25년 이력 검증 완료)</b>',
        labels={'기업명': '기업명', '표시높이': '발급 상태', '발급상태': '구분'},
        color_discrete_map=color_map
    )

    fig.update_layout(
        font_family='Apple SD Gothic Neo',
        yaxis=dict(showticklabels=False, range=[0, 1.25]),
        xaxis=dict(tickangle=-35),
        legend=dict(title="보고서 발급 구분"),
        margin=dict(l=40, r=40, t=60, b=80)
    )

    fig.update_traces(textposition='inside')
    fig.for_each_annotation(lambda a: a.update(text=f"<b>{a.text.split('=')[-1]}</b>"))
    
    fig.show()

In [29]:
import pandas as pd
import plotly.express as px

# 1. 엑셀 파일 로드
excel_path = '/Users/lusia/Personal/13. DA Bootcamp/04. Project/2. ESG/01_데이터셋/260827_전기전자업 전체 업종.xlsx'
df_excel = pd.read_excel(excel_path, header=None)
df_excel.columns = ['기업명', '기업코드']
df_excel['기업명'] = df_excel['기업명'].astype(str).str.strip()

# 2. 한국표준산업분류(KSIC) C26 vs C28 분류 함수 정의
def assign_ksic(company_name):
    # C26: 전자부품, 컴퓨터, 영상, 음향 및 통신장비 제조업 (반도체, PCB, 디스플레이, 통신/영상기기 등)
    c26_keywords = [
        'DB하이텍', 'KEC', 'SK하이닉스', '삼성전자', '네패스', 'SF반도체', '해성디에스', '하나마이크론', 
        '어보브반도체', '시그네틱스', '제주반도체', '텔레칩스', '칩스앤미디어', '리노공업', '동진쎄미켐', 'SOLUM', '솔브레인',
        'LG디스플레이', 'LX세미콘', '일진디스플', '파워넷', '선익시스템', '야스', '인포뱅크', '아이씨디', '덕산네오룩스', '코세스',
        'LG이노텍', '삼성전기', '심텍', '대덕전자', '코리아써키트', '이수페타시스', '한화시스템', '주연테크'
    ]
    
    # C28: 전기장비 제조업 (배터리/이차전지, 전선, 발전기, 조명, 가전제품 등)
    c28_keywords = [
        'LG에너지솔루션', '삼성SDI', 'SK아이이테크놀로지', '엘앤에프', '포스코퓨처엠', '에코프로비엠', 'DN오토모티브', 'LS머트리얼즈', '모베이스전자', '현대모비스', 'HL만도',
        'HD현대일렉트릭', 'LS일렉트릭', 'LS전선', '대한전선', '가온전선', '일진전기', '효성중공업', '제룡전기', '광명전기', '보성파워텍', '서전기전', 'HD현대에너지솔루션', 'KH 필룩스',
        'LG전자', '위니아', '쿠쿠홈시스', '쿠쿠홀딩스', '신일전자', '파セ', '오텍', '위닉스', 'PN풍년', '두산퓨얼셀'
    ]

    if any(k in company_name for k in c28_keywords):
        return 'C28: 전기장비 제조업'
    else:
        return 'C26: 전자부품, 컴퓨터, 영상, 음향 및 통신장비 제조업'

# 3. 분류 적용
df_excel['KSIC_분류'] = df_excel['기업명'].apply(assign_ksic)

# 4. 분류별 기업 수 집계
ksic_summary = df_excel.groupby('KSIC_분류')['기업명'].agg(
    기업수='nunique',
    기업목록=lambda x: ", ".join(x)
).reset_index()

# ----------------------------------------------------
# [EDA 결과 출력]
# ----------------------------------------------------
print("=" * 80)
print("■ 1. 한국표준산업분류(C26 / C28) 대분류별 기업 수 집계")
print("=" * 80)
print(ksic_summary[['KSIC_분류', '기업수']].to_string(index=False))
print(f"\n총 포함 기업 수: {ksic_summary['기업수'].sum()}개사")

print("\n" + "=" * 80)
print("■ 2. 세부 분류별 포함 기업 목록")
print("=" * 80)
for _, row in ksic_summary.iterrows():
    print(f"\n▶ [{row['KSIC_분류']}] - 총 {row['기업수']}개사")
    print(f"   기업 목록: {row['기업목록']}")

# ----------------------------------------------------
# Plotly 시각화 (C26 vs C28 분포)
# ----------------------------------------------------
fig = px.bar(
    ksic_summary,
    x='KSIC_분류',
    y='기업수',
    text='기업수',
    color='KSIC_분류',
    title='<b>엑셀 Target 기업 C26 vs C28 산업분류별 기업 수 분포</b>',
    labels={'KSIC_분류': '산업분류', '기업수': '기업 수 (개사)'},
    color_discrete_sequence=['#1D3557', '#457B9D']
)

fig.update_traces(texttemplate='%{text}개사', textposition='outside')
fig.update_layout(
    font_family='Apple SD Gothic Neo',
    showlegend=False,
    yaxis=dict(range=[0, ksic_summary['기업수'].max() * 1.25])
)
fig.show()

■ 1. 한국표준산업분류(C26 / C28) 대분류별 기업 수 집계
                          KSIC_분류  기업수
C26: 전자부품, 컴퓨터, 영상, 음향 및 통신장비 제조업   82
                    C28: 전기장비 제조업   19

총 포함 기업 수: 101개사

■ 2. 세부 분류별 포함 기업 목록

▶ [C26: 전자부품, 컴퓨터, 영상, 음향 및 통신장비 제조업] - 총 82개사
   기업 목록: DB하이텍, KEC, LG디스플레이, LG이노텍, LX세미콘, RFHIC, SFA반도체, SK하이닉스, 가온칩스, 경동나비엔, 경인전자, 광전자, 금호전기, 네패스, 다원시스, 대덕전자, 대동전자, 대원전선, 대주전자재료, 더블유씨피, 덕산네오룩스, 드림텍, 롯데에너지머티리얼즈, 리노공업, 비에이치, 삼성전기, 삼성전자, 삼영전자공업, 삼화전기, 삼화전자공업, 삼화콘덴서공업, 서울반도체, 서진시스템, 선도전기, 성문전자, 세방전지, 솔루스첨단소재, 솔루엠, 시그네틱스, 시노펙스, 신도리코, 신성델타테크, 신성이엔지, 심텍, 써니전자, 씨아이테크, 아남전자, 아이티엠반도체, 에스앤에스텍, 에스피지, 에이엔피, 에코프로머티, 에프에스티, 엑시큐어하이트론, 엘에스일렉트릭, 엠씨넥스, 와이투솔루션, 유티아이, 이녹스첨단소재, 이수페타시스, 이엠텍, 인탑스, 인터플렉스, 인텔리안테크, 일진디스플, 자화전자, 제이앤티씨, 제주반도체, 주성코퍼레이션, 주연테크, 칩스앤미디어, 케이엠더블유, 코리아써키트, 티에이치엔, 파트론, 하나마이크론, 하나머티리얼즈, 한국단자공업, 한솔테크닉스, 한화시스템, 해성디에스, 휴니드테크놀러지스

▶ [C28: 전기장비 제조업] - 총 19개사
   기업 목록: DN오토모티브, HD현대에너지솔루션, HD현대일렉트릭, KH 필룩스, LG에너지솔루션, LG전자, LS머트리얼즈, SK아이이테크놀로지, 가온전선, 광명전기, 대한전선, 두산퓨얼셀, 삼성SDI, 에코프로비엠, 엘앤에프, 일진전기, 제룡전기, 

In [30]:
import pandas as pd
import plotly.express as px
import plotly.io as pio

# 1. Plotly 기본 템플릿 설정
pio.renderers.default = "notebook"
pio.templates.default = "plotly_white"

# 2. 파일 로드
excel_path = '/Users/lusia/Personal/13. DA Bootcamp/04. Project/2. ESG/01_데이터셋/260827_전기전자업 전체 업종.xlsx'
csv_path = '/Users/lusia/Personal/13. DA Bootcamp/04. Project/2. ESG/01_데이터셋/KCGS_평가등급_2022-2025_Long_merged.csv'

df_excel = pd.read_excel(excel_path, header=None)
df_excel.columns = ['기업명', '기업코드']
df_excel['기업명'] = df_excel['기업명'].astype(str).str.strip()

df_kcgs = pd.read_csv(csv_path)
df_kcgs['기업명'] = df_kcgs['기업명'].astype(str).str.strip()

# ----------------------------------------------------
# 3. C26 vs C28 산업분류 매핑
# ----------------------------------------------------
def assign_ksic(company_name):
    # C28 키워드
    c28_keywords = [
        'LG에너지솔루션', '삼성SDI', 'SK아이이테크놀로지', '엘앤에프', '포스코퓨처엠', '에코프로비엠', 
        'DN오토모티브', 'LS머트리얼즈', '모베이스전자', '현대모비스', 'HL만도', 'HD현대일렉트릭', 
        'LS일렉트릭', 'LS전선', '대한전선', '가온전선', '일진전기', '효성중공업', '제룡전기', 
        '광명전기', '보성파워텍', '서전기전', 'HD현대에너지솔루션', 'KH 필룩스', 'LG전자', 
        '위니아', '쿠쿠홈시스', '쿠쿠홀딩스', '신일전자', '파セ', '오텍', '위닉스', 'PN풍년', '두산퓨얼셀'
    ]
    if any(k in company_name for k in c28_keywords):
        return 'C28: 전기장비 제조업'
    return 'C26: 전자부품, 컴퓨터, 영상, 음향 및 통신장비 제조업'

df_excel['KSIC_분류'] = df_excel['기업명'].apply(assign_ksic)

# ----------------------------------------------------
# 4. 데이터 통합 및 2025년 최신 등급 추출
# ----------------------------------------------------
df_merged = pd.merge(df_kcgs, df_excel[['기업명', 'KSIC_분류']], on='기업명', how='inner')

# C26 업종 필터링
c26_full = df_merged[df_merged['KSIC_분류'].str.startswith('C26')].copy()

# 2025년 최신 평가 등급 기준 추출 (2025년 데이터가 없으면 최근 연도 기준)
c26_2025 = c26_full.sort_values('평가년도').groupby('기업명').last().reset_index()

grade_order = ['A+', 'A', 'B+', 'B', 'C', 'D']
c26_2025['ESG등급'] = pd.Categorical(c26_2025['ESG등급'], categories=grade_order, ordered=True)

# ----------------------------------------------------
# 5. [EDA] 모범 벤치마킹 (A 이상) & 컨설팅 타겟 (하위 10개사) 추출
# ----------------------------------------------------

# (1) 모범 벤치마킹 기업 (ESG 등급 A+ 및 A)
benchmark_df = c26_2025[c26_2025['ESG등급'].isin(['A+', 'A'])].sort_values(['ESG등급', '기업명'])
benchmark_df['구분'] = '모범 벤치마킹'

# (2) 컨설팅 필요 하위 10개사 (낮은 등급 순서 정렬)
# D -> C -> B -> B+ -> A -> A+ 순으로 가장 낮은 기업 10개 추출
consulting_df = c26_2025.sort_values(['ESG등급', '기업명'], ascending=[False, True]).head(10).sort_values(['ESG등급', '기업명'])
consulting_df['구분'] = '컨설팅 대상 (하위 10개사)'

# ----------------------------------------------------
# [EDA 결과 출력]
# ----------------------------------------------------
print("=" * 80)
print(f"■ C26 업종 전체 분석 기업 수: 총 {len(c26_2025)}개사")
print("=" * 80)

print("\n1. [모범 벤치마킹 대상] (ESG 등급 A 이상)")
print("-" * 50)
print(benchmark_df[['기업명', 'ESG등급', '평가년도']].to_string(index=False))
print(f"총 {len(benchmark_df)}개사")

print("\n2. [컨설팅 타겟 대상] (하위 10개사)")
print("-" * 50)
print(consulting_df[['기업명', 'ESG등급', '평가년도']].to_string(index=False))

# ----------------------------------------------------
# [시각화 1] C26 업종 모범 벤치마킹 vs 컨설팅 대상 등급 분포 비교 Bar Chart
# ----------------------------------------------------
eda_combined = pd.concat([benchmark_df, consulting_df])

fig1 = px.bar(
    eda_combined,
    x='기업명',
    y='ESG등급',
    color='구분',
    text='ESG등급',
    title='<b>[C26 업종] 모범 벤치마킹 기업 vs 컨설팅 타겟 하위 10개사 ESG 등급 현황</b>',
    labels={'기업명': '기업명', 'ESG등급': 'ESG 통합 등급'},
    color_discrete_map={
        '모범 벤치마킹': '#1D3557', 
        '컨설팅 대상 (하위 10개사)': '#E63946'
    }
)

fig1.update_traces(textposition='outside')
fig1.update_layout(
    font_family='Apple SD Gothic Neo',
    xaxis=dict(tickangle=-45),
    yaxis=dict(categoryorder='array', categoryarray=grade_order[::-1]), # D부터 A+ 순으로 축 정렬
    margin=dict(l=40, r=40, t=60, b=80)
)

fig1.show()

# ----------------------------------------------------
# [시각화 2] C26 전체 등급 구조 및 타겟 그룹 비중 Pie Chart
# ----------------------------------------------------
c26_summary = c26_2025['ESG등급'].value_counts().reindex(grade_order).fillna(0).reset_index()
c26_summary.columns = ['ESG등급', '기업수']

fig2 = px.pie(
    c26_summary,
    names='ESG등급',
    values='기업수',
    title='<b>[C26 업종] 전체 ESG 등급 분포 비중 (2025년 기준)</b>',
    color='ESG등급',
    color_discrete_map={
        'A+': '#1D3557', 'A': '#457B9D', 'B+': '#A8DADC',
        'B': '#F4A261', 'C': '#E76F51', 'D': '#E63946'
    },
    hole=0.4
)

fig2.update_traces(textinfo='label+value+percent')
fig2.update_layout(font_family='Apple SD Gothic Neo')

fig2.show()

■ C26 업종 전체 분석 기업 수: 총 82개사

1. [모범 벤치마킹 대상] (ESG 등급 A 이상)
--------------------------------------------------
    기업명 ESG등급  평가년도
LG디스플레이     A  2025
  LG이노텍     A  2025
  LX세미콘     A  2025
 SK하이닉스     A  2025
   삼성전기     A  2025
   삼성전자     A  2025
     심텍     A  2025
엘에스일렉트릭     A  2025
 한솔테크닉스     A  2025
  한화시스템     A  2025
  해성디에스     A  2025
총 11개사

2. [컨설팅 타겟 대상] (하위 10개사)
--------------------------------------------------
  기업명 ESG등급  평가년도
  KEC     D  2025
RFHIC     D  2025
 가온칩스     D  2025
 경인전자     D  2025
  광전자     D  2025
 금호전기     D  2025
 다원시스     D  2025
 대동전자     D  2025
 대원전선     D  2025
더블유씨피     D  2025


In [31]:
import pandas as pd
import plotly.express as px
import plotly.io as pio

# 1. Plotly 기본 템플릿 설정
pio.renderers.default = "notebook"
pio.templates.default = "plotly_white"

# 2. 파일 로드
excel_path = '/Users/lusia/Personal/13. DA Bootcamp/04. Project/2. ESG/01_데이터셋/260827_전기전자업 전체 업종.xlsx'
csv_path = '/Users/lusia/Personal/13. DA Bootcamp/04. Project/2. ESG/01_데이터셋/KCGS_평가등급_2022-2025_Long_merged.csv'

df_excel = pd.read_excel(excel_path, header=None)
df_excel.columns = ['기업명', '기업코드']
df_excel['기업명'] = df_excel['기업명'].astype(str).str.strip()

df_kcgs = pd.read_csv(csv_path)
df_kcgs['기업명'] = df_kcgs['기업명'].astype(str).str.strip()

# ----------------------------------------------------
# 3. C26 vs C28 산업분류 매핑
# ----------------------------------------------------
def assign_ksic(company_name):
    # C28 키워드 (전기장비 제조업: 배터리, 전선, 발전기, 조명, 가전 등)
    c28_keywords = [
        'LG에너지솔루션', '삼성SDI', 'SK아이이테크놀로지', '엘앤에프', '포스코퓨처엠', '에코프로비엠', 
        'DN오토모티브', 'LS머트리얼즈', '모베이스전자', '현대모비스', 'HL만도', 'HD현대일렉트릭', 
        'LS일렉트릭', 'LS전선', '대한전선', '가온전선', '일진전기', '효성중공업', '제룡전기', 
        '광명전기', '보성파워텍', '서전기전', 'HD현대에너지솔루션', 'KH 필룩스', 'LG전자', 
        '위니아', '쿠쿠홈시스', '쿠쿠홀딩스', '신일전자', '파セ', '오텍', '위닉스', 'PN풍년', '두산퓨얼셀'
    ]
    if any(k in company_name for k in c28_keywords):
        return 'C28: 전기장비 제조업'
    return 'C26: 전자부품, 컴퓨터, 영상, 음향 및 통신장비 제조업'

df_excel['KSIC_분류'] = df_excel['기업명'].apply(assign_ksic)

# ----------------------------------------------------
# 4. 데이터 통합 및 2025년 최신 등급 추출
# ----------------------------------------------------
df_merged = pd.merge(df_kcgs, df_excel[['기업명', 'KSIC_분류']], on='기업명', how='inner')

# C28 업종 필터링
c28_full = df_merged[df_merged['KSIC_분류'].str.startswith('C28')].copy()

# 2025년 최신 평가 등급 기준 추출 (2025년 데이터가 없으면 최근 연도 기준)
c28_2025 = c28_full.sort_values('평가년도').groupby('기업명').last().reset_index()

grade_order = ['A+', 'A', 'B+', 'B', 'C', 'D']
c28_2025['ESG등급'] = pd.Categorical(c28_2025['ESG등급'], categories=grade_order, ordered=True)

# ----------------------------------------------------
# 5. [EDA] C28 모범 벤치마킹 (A 이상) & 컨설팅 타겟 (하위 10개사) 추출
# ----------------------------------------------------

# (1) 모범 벤치마킹 기업 (ESG 등급 A+ 및 A)
benchmark_c28 = c28_2025[c28_2025['ESG등급'].isin(['A+', 'A'])].sort_values(['ESG등급', '기업명'])
benchmark_c28['구분'] = '모범 벤치마킹'

# (2) 컨설팅 필요 하위 10개사 (낮은 등급 순서 정렬)
consulting_c28 = c28_2025.sort_values(['ESG등급', '기업명'], ascending=[False, True]).head(10).sort_values(['ESG등급', '기업명'])
consulting_c28['구분'] = '컨설팅 대상 (하위 10개사)'

# ----------------------------------------------------
# [EDA 결과 출력]
# ----------------------------------------------------
print("=" * 80)
print(f"■ C28 업종 전체 분석 기업 수: 총 {len(c28_2025)}개사")
print("=" * 80)

print("\n1. [C28 모범 벤치마킹 대상] (ESG 등급 A 이상)")
print("-" * 50)
print(benchmark_c28[['기업명', 'ESG등급', '평가년도']].to_string(index=False))
print(f"총 {len(benchmark_c28)}개사")

print("\n2. [C28 컨설팅 타겟 대상] (하위 10개사)")
print("-" * 50)
print(consulting_c28[['기업명', 'ESG등급', '평가년도']].to_string(index=False))

# ----------------------------------------------------
# [시각화 1] C28 업종 모범 벤치마킹 vs 컨설팅 대상 등급 분포 비교 Bar Chart
# ----------------------------------------------------
eda_combined_c28 = pd.concat([benchmark_c28, consulting_c28])

fig1 = px.bar(
    eda_combined_c28,
    x='기업명',
    y='ESG등급',
    color='구분',
    text='ESG등급',
    title='<b>[C28 업종] 모범 벤치마킹 기업 vs 컨설팅 타겟 하위 10개사 ESG 등급 현황</b>',
    labels={'기업명': '기업명', 'ESG등급': 'ESG 통합 등급'},
    color_discrete_map={
        '모범 벤치마킹': '#1D3557', 
        '컨설팅 대상 (하위 10개사)': '#E63946'
    }
)

fig1.update_traces(textposition='outside')
fig1.update_layout(
    font_family='Apple SD Gothic Neo',
    xaxis=dict(tickangle=-45),
    yaxis=dict(categoryorder='array', categoryarray=grade_order[::-1]), # D부터 A+ 순으로 축 정렬
    margin=dict(l=40, r=40, t=60, b=80)
)

fig1.show()

# ----------------------------------------------------
# [시각화 2] C28 전체 등급 구조 및 타겟 그룹 비중 Pie Chart
# ----------------------------------------------------
c28_summary = c28_2025['ESG등급'].value_counts().reindex(grade_order).fillna(0).reset_index()
c28_summary.columns = ['ESG등급', '기업수']

fig2 = px.pie(
    c28_summary,
    names='ESG등급',
    values='기업수',
    title='<b>[C28 업종] 전체 ESG 등급 분포 비중 (2025년 기준)</b>',
    color='ESG등급',
    color_discrete_map={
        'A+': '#1D3557', 'A': '#457B9D', 'B+': '#A8DADC',
        'B': '#F4A261', 'C': '#E76F51', 'D': '#E63946'
    },
    hole=0.4
)

fig2.update_traces(textinfo='label+value+percent')
fig2.update_layout(font_family='Apple SD Gothic Neo')

fig2.show()

■ C28 업종 전체 분석 기업 수: 총 19개사

1. [C28 모범 벤치마킹 대상] (ESG 등급 A 이상)
--------------------------------------------------
       기업명 ESG등급  평가년도
  HD현대일렉트릭    A+  2025
HD현대에너지솔루션     A  2025
  LG에너지솔루션     A  2025
      LG전자     A  2025
SK아이이테크놀로지     A  2025
      대한전선     A  2025
     두산퓨얼셀     A  2025
     삼성SDI     A  2025
    포스코퓨처엠     A  2025
총 9개사

2. [C28 컨설팅 타겟 대상] (하위 10개사)
--------------------------------------------------
    기업명 ESG등급  평가년도
   가온전선    B+  2025
 에코프로비엠    B+  2025
   엘앤에프    B+  2025
LS머트리얼즈     B  2025
  효성중공업     B  2025
DN오토모티브     C  2025
   제룡전기     C  2025
 KH 필룩스     D  2025
   광명전기     D  2025
   일진전기     D  2025


In [32]:
import pandas as pd

# 1. 엑셀 파일 로드 (전기전자업 Target 기업 101개사)
excel_path = '/Users/lusia/Personal/13. DA Bootcamp/04. Project/2. ESG/01_데이터셋/260827_전기전자업 전체 업종.xlsx'
df_excel = pd.read_excel(excel_path, header=None)
df_excel.columns = ['기업명', '기업코드']
df_excel['기업명'] = df_excel['기업명'].astype(str).str.strip()

# 2. 281(전동기, 발전기 및 전기 변환·공급·제어 장치 제조업) 하위 4자리 분류 함수
def assign_ksic_281_detail(company_name):
    # [2811] 전동기, 발전기 및 전기 변환장치 제조업
    # (전동기, 발전기, 변압기, 에너지저장장치(ESS), 기타 전기 변환장치)
    c2811_keywords = [
        'HD현대일렉트릭', '효성중공업', '제룡전기', 'HD현대에너지솔루션', 
        '두산퓨얼셀', '와이투솔루션', '솔루엠', 'SOLUM'
    ]
    
    # [2812] 전기 공급 및 제어장치 제조업
    # (전기회로 개폐·보호장치, 접속장치, 배전반 및 전기 자동제어반)
    c2812_keywords = [
        'LS일렉트릭', '엘에스일렉트릭', '광명전기', '보성파워텍', 
        '서전기전', '선도전기', '가온전선' # 제어반 및 배전 관련 기업
    ]

    if any(k in company_name for k in c2811_keywords):
        return '2811. 전동기, 발전기 및 전기 변환장치 제조업'
    elif any(k in company_name for k in c2812_keywords):
        return '2812. 전기 공급 및 제어장치 제조업'
    else:
        return '기타 C28/C26 세부 업종 (전선, 배터리, 반도체 등)'

# 3. 분류 적용
df_excel['KSIC_4자리_코드'] = df_excel['기업명'].apply(assign_ksic_281_detail)

# 4. 2811 및 2812 해당 기업만 필터링
target_281_df = df_excel[df_excel['KSIC_4자리_코드'].str.startswith('281')].copy()

# ----------------------------------------------------
# [EDA 결과 출력]
# ----------------------------------------------------
print("=" * 80)
print("■ [코드 281 하위 4자리 기준] 기업명 추출 결과")
print("=" * 80)

summary_281 = target_281_df.groupby('KSIC_4자리_코드')['기업명'].agg(
    기업수='count',
    기업명_리스트=lambda x: ", ".join(x)
).reset_index()

for _, row in summary_281.iterrows():
    print(f"\n▶ [{row['KSIC_4자리_코드']}] - 총 {row['기업수']}개사")
    print(f"   기업 목록: {row['기업명_리스트']}")

print("\n" + "=" * 80)

■ [코드 281 하위 4자리 기준] 기업명 추출 결과

▶ [2811. 전동기, 발전기 및 전기 변환장치 제조업] - 총 7개사
   기업 목록: HD현대에너지솔루션, HD현대일렉트릭, 두산퓨얼셀, 솔루엠, 와이투솔루션, 제룡전기, 효성중공업

▶ [2812. 전기 공급 및 제어장치 제조업] - 총 4개사
   기업 목록: 가온전선, 광명전기, 선도전기, 엘에스일렉트릭

